# Improved, auditable candidate data cleaning

This is a separate end-to-end notebook for Andhra Pradesh, Bihar and Gujarat in 2014, 2019 and 2024. It reads from `ocr_data/` first and falls back to `data/`; it never edits the existing notebooks or input CSVs. Outputs are written to `improved_cleaned_data/`.

Original strings remain available. Unknown or unparseable amounts stay missing; only explicit `Nil` becomes zero. Suspected outliers and duplicates are flagged for human review, never silently changed or removed.

## 1. Raw-file profile, cleaning and outputs

Run the following cell from the project folder. It reports every input's shape, dtypes, missing counts and field meanings, then builds the separate clean outputs.

In [2]:
from pathlib import Path
import re, unicodedata
import numpy as np, pandas as pd

ROOT=Path.cwd(); OUT=ROOT/"improved_cleaned_data"; OUT.mkdir(exist_ok=True)
STATES=["andhra","bihar","gujarat"]; YEARS=[2014,2019,2024]
def input_file(s,y):
    for folder in ["ocr_data","data"]:
        p=ROOT/folder/f"{s}_{y}.csv"
        if p.exists(): return p
    raise FileNotFoundError(f"Missing {s}_{y}.csv")
def name_key(x):
    if pd.isna(x): return pd.NA
    x=unicodedata.normalize("NFKC",str(x)).strip()
    x=re.sub(r"^(?:(?:adv|dr|mr|mrs|ms|shri|smt|prof)\.?\s+)+","",x,flags=re.I)
    x=re.sub(r"\s+(?:s/o|d/o|w/o|h/o|c/o)\.?\s+.*$","",x,flags=re.I)
    x=re.sub(r"\([^)]*\)|[,;.]"," ",x)
    return re.sub(r"\s+"," ",x).strip().casefold() or pd.NA
def money(x):
    if pd.isna(x) or not str(x).strip(): return np.nan,"missing"
    x=unicodedata.normalize("NFKC",str(x)).strip()
    if re.search(r"\bnil\b",x,re.I): return 0,"explicit_nil_zero"
    m=re.search(r"\brs\.?\s*([0-9][0-9,]*(?:\.[0-9]+)?)",x,re.I)
    if not m: return np.nan,"unparsed_review"
    try: n=float(m.group(1).replace(",",""))
    except ValueError: return np.nan,"unparsed_review"
    if not np.isfinite(n) or n<0: return np.nan,"invalid_review"
    return (int(n) if n.is_integer() else n),("parsed" if n.is_integer() else "decimal_review")
def age(x):
    if pd.isna(x) or not str(x).strip(): return np.nan,"missing"
    m=re.search(r"\d{1,3}",str(x))
    if not m: return np.nan,"unparsed_review"
    n=int(m.group())
    return (n,"parsed") if 25<=n<=100 else (np.nan,"implausible_review")
def number(x):
    if pd.isna(x) or not str(x).strip(): return np.nan,"missing"
    m=re.search(r"\d+",str(x))
    return (int(m.group()),"parsed") if m else (np.nan,"unparsed_review")

# Profile every raw file before transformations.
meanings={"Sno.":"Source row number, not a stable identifier.","Candidate":"Candidate name from MyNeta.",
"Party":"Party label from MyNeta.","Criminal Cases":"Declared criminal-case count.",
"Education":"Declared education category.","Age":"Declared age in years.",
"Total Assets":"Declared assets text; parsed output is nominal INR.",
"Liabilities":"Declared liabilities text; parsed output is nominal INR."}
raws={}; prof=[]
for s in STATES:
 for y in YEARS:
    p=input_file(s,y)
    d=pd.read_csv(p,dtype={"Candidate":"string","Party":"string","Total Assets":"string","Liabilities":"string"})
    raws[s,y]=(p,d)
    for c in d:
        prof.append({"source_file":str(p.relative_to(ROOT)),"state":s.title(),"year":y,"rows":len(d),
        "columns":len(d.columns),"column":c,"dtype_before":str(d[c].dtype),
        "missing_before":int(d[c].isna().sum()),"meaning":meanings.get(c,"Verify meaning on source page.")})
profile=pd.DataFrame(prof); display(profile); profile.to_csv(OUT/"raw_file_profile.csv",index=False)

# Keep originals, create a matching aid and parse statuses. No inferred corrections.
parts=[]
for (s,y),(p,raw) in raws.items():
 d=raw.copy(deep=True); d.insert(0,"State",s.title()); d.insert(1,"Election Year",y)
 d["source_file"]=str(p.relative_to(ROOT))
 for c in ["Candidate","Party","Education","Age","Total Assets","Liabilities"]:
    if c in d: d[c+"_source"]=d[c].astype("string")
 d["Candidate_match_key"]=d.Candidate.map(name_key)
 av=d.Age.map(age); d["Age_years"]=[v[0] for v in av]; d["Age_parse_status"]=[v[1] for v in av]
 for c,o in [("Total Assets","Total_Assets_INR"),("Liabilities","Liabilities_INR")]:
    v=d[c].map(money); d[o]=[z[0] for z in v]; d[o+"_parse_status"]=[z[1] for z in v]
 if "Criminal Cases" in d:
    v=d["Criminal Cases"].map(number); d["Criminal_Cases_count"]=[z[0] for z in v]
    d["Criminal_Cases_parse_status"]=[z[1] for z in v]
 # Flag, do not automatically remove: no stable ID or constituency in these files.
 key=[c for c in ["Candidate_match_key","Party","Age_years","Total_Assets_INR","Liabilities_INR","Education"] if c in d]
 d["duplicate_record_flag"]=d.duplicated(subset=key,keep=False); parts.append(d)
clean=pd.concat(parts,ignore_index=True)

# 1.5*IQR is a review flag only; no capping/deletion.
for metric in ["Total_Assets_INR","Liabilities_INR"]:
 flag=metric+"_iqr_outlier_flag"; clean[flag]=False
 for _,ix in clean.groupby(["State","Election Year"]).groups.items():
    v=clean.loc[ix,metric].dropna()
    if len(v)>=4:
        q1,q3=v.quantile([.25,.75]); clean.loc[ix,flag]=clean.loc[ix,metric].gt(q3+1.5*(q3-q1)).fillna(False)
clean["manual_review_flag"]=(
 clean.Age_parse_status.str.contains("review|unparsed",case=False,na=False)
 |clean.Total_Assets_INR_parse_status.str.contains("review|unparsed",case=False,na=False)
 |clean.Liabilities_INR_parse_status.str.contains("review|unparsed",case=False,na=False)
 |clean.duplicate_record_flag|clean.Total_Assets_INR_iqr_outlier_flag|clean.Liabilities_INR_iqr_outlier_flag)

summ=[]
for (s,y),(p,raw) in raws.items():
 z=clean[(clean.State==s.title())&(clean["Election Year"]==y)]
 summ.append({"state":s.title(),"year":y,"file":str(p.relative_to(ROOT)),"rows_before":len(raw),
 "rows_after":len(z),"rows_removed":len(raw)-len(z),"duplicate_rows_flagged":int(z.duplicate_record_flag.sum()),
 "asset_missing_or_unparsed":int(z.Total_Assets_INR.isna().sum()),
 "assets_nil_as_zero":int(z.Total_Assets_INR_parse_status.eq("explicit_nil_zero").sum()),
 "liability_missing_or_unparsed":int(z.Liabilities_INR.isna().sum()),
 "age_missing_or_unparsed":int(z.Age_years.isna().sum()),
 "asset_iqr_flags":int(z.Total_Assets_INR_iqr_outlier_flag.sum()),
 "liability_iqr_flags":int(z.Liabilities_INR_iqr_outlier_flag.sum()),
 "manual_review_rows":int(z.manual_review_flag.sum())})
summary=pd.DataFrame(summ); display(summary); summary.to_csv(OUT/"cleaning_summary.csv",index=False)
missing=clean.groupby(["Election Year","State"]).agg(rows=("Candidate","size"),
 missing_candidate=("Candidate",lambda x:x.isna().sum()),missing_age=("Age_years",lambda x:x.isna().sum()),
 missing_assets=("Total_Assets_INR",lambda x:x.isna().sum()),
 missing_liabilities=("Liabilities_INR",lambda x:x.isna().sum())).reset_index()
display(missing); missing.to_csv(OUT/"missingness_by_election_state.csv",index=False)

review_cols=[c for c in ["State","Election Year","Sno.","Candidate_source","Candidate_match_key","Party_source",
"Age_source","Age_years","Age_parse_status","Total Assets_source","Total_Assets_INR","Total_Assets_INR_parse_status",
"Liabilities_source","Liabilities_INR","Liabilities_INR_parse_status","Total_Assets_INR_iqr_outlier_flag",
"Liabilities_INR_iqr_outlier_flag","duplicate_record_flag","source_file"] if c in clean]
clean.loc[clean.manual_review_flag,review_cols].to_csv(OUT/"manual_review.csv",index=False)

# Save a separate Parquet file for each state and election year.
# The combined table stays in memory for profiling; no combined Parquet is written.
STATE_YEAR_DIR=OUT/"state_year"
STATE_YEAR_DIR.mkdir(exist_ok=True)
analysis=clean.drop(columns=["Age","Total Assets","Liabilities"],errors="ignore")
dtype_rows=[]
written_rows=0
for (state,year),(_,raw) in raws.items():
    part=analysis[(analysis.State==state.title())&(analysis["Election Year"]==year)].copy()
    target=STATE_YEAR_DIR/f"{state}_{year}.parquet"
    part.to_parquet(target,index=False)
    back=pd.read_parquet(target)
    assert len(back)==len(part)==len(raw), f"Row count mismatch for {state} {year}"
    assert (back.Total_Assets_INR.dropna()>=0).all(), f"Negative assets in {target}"
    assert (back.Liabilities_INR.dropna()>=0).all(), f"Negative liabilities in {target}"
    dtype_rows.extend({"state":state.title(),"year":year,"file":target.name,
                       "column":column,"dtype_after_readback":str(back[column].dtype)}
                      for column in back.columns)
    written_rows += len(back)
    print(f"Saved {len(back)} rows: {target}")
dtype=pd.DataFrame(dtype_rows)
display(dtype)
dtype.to_csv(OUT/"dtype_check_after_parquet_readback.csv",index=False)
print(f"Saved {written_rows} rows across {len(raws)} state/year Parquet files.")
print(f"Manual review queue: {int(clean.manual_review_flag.sum())}")


,source_file,state,year,rows,columns,column,dtype_before,missing_before,meaning
0,ocr_data\andhra_2014.csv,Andhra,2014,329,8,Sno.,int64,0,"Source row number, not a stable identifier."
1,ocr_data\andhra_2014.csv,Andhra,2014,329,8,Candidate,string,0,Candidate name from MyNeta.
2,ocr_data\andhra_2014.csv,Andhra,2014,329,8,Party,string,0,Party label from MyNeta.
3,ocr_data\andhra_2014.csv,Andhra,2014,329,8,Criminal Cases,int64,0,Declared criminal-case count.
4,ocr_data\andhra_2014.csv,Andhra,2014,329,8,Education,str,0,Declared education category.
...,...,...,...,...,...,...,...,...,...
67,ocr_data\gujarat_2024.csv,Gujarat,2024,266,8,Criminal Cases,int64,0,Declared criminal-case count.
68,ocr_data\gujarat_2024.csv,Gujarat,2024,266,8,Education,str,0,Declared education category.
69,ocr_data\gujarat_2024.csv,Gujarat,2024,266,8,Age,int64,0,Declared age in years.
70,ocr_data\gujarat_2024.csv,Gujarat,2024,266,8,Total Assets,string,0,Declared assets text; parsed output is nominal...


,state,year,file,rows_before,rows_after,rows_removed,duplicate_rows_flagged,asset_missing_or_unparsed,assets_nil_as_zero,liability_missing_or_unparsed,age_missing_or_unparsed,asset_iqr_flags,liability_iqr_flags,manual_review_rows
0,Andhra,2014,ocr_data\andhra_2014.csv,329,329,0,0,0,8,0,7,52,53,78
1,Andhra,2019,ocr_data\andhra_2019.csv,314,314,0,0,0,5,0,3,49,58,73
2,Andhra,2024,ocr_data\andhra_2024.csv,450,450,0,2,0,11,0,0,68,76,95
3,Bihar,2014,ocr_data\bihar_2014.csv,605,605,0,0,0,2,0,6,83,109,156
4,Bihar,2019,ocr_data\bihar_2019.csv,619,619,0,0,0,2,0,0,79,97,138
5,Bihar,2024,ocr_data\bihar_2024.csv,496,496,0,4,0,0,0,0,63,81,117
6,Gujarat,2014,ocr_data\gujarat_2014.csv,333,333,0,0,0,1,0,5,45,58,77
7,Gujarat,2019,ocr_data\gujarat_2019.csv,371,371,0,4,0,1,0,0,59,70,97
8,Gujarat,2024,ocr_data\gujarat_2024.csv,266,266,0,2,0,0,0,0,44,40,61


,Election Year,State,rows,missing_candidate,missing_age,missing_assets,missing_liabilities
0,2014,Andhra,329,0,7,0,0
1,2014,Bihar,605,0,6,0,0
2,2014,Gujarat,333,0,5,0,0
3,2019,Andhra,314,0,3,0,0
4,2019,Bihar,619,0,0,0,0
5,2019,Gujarat,371,0,0,0,0
6,2024,Andhra,450,0,0,0,0
7,2024,Bihar,496,0,0,0,0
8,2024,Gujarat,266,0,0,0,0


Saved 329 rows: d:\IITM\Academics\Sem 3\DCV\Projects\MPs-assets\improved_cleaned_data\state_year\andhra_2014.parquet
Saved 314 rows: d:\IITM\Academics\Sem 3\DCV\Projects\MPs-assets\improved_cleaned_data\state_year\andhra_2019.parquet
Saved 450 rows: d:\IITM\Academics\Sem 3\DCV\Projects\MPs-assets\improved_cleaned_data\state_year\andhra_2024.parquet
Saved 605 rows: d:\IITM\Academics\Sem 3\DCV\Projects\MPs-assets\improved_cleaned_data\state_year\bihar_2014.parquet
Saved 619 rows: d:\IITM\Academics\Sem 3\DCV\Projects\MPs-assets\improved_cleaned_data\state_year\bihar_2019.parquet
Saved 496 rows: d:\IITM\Academics\Sem 3\DCV\Projects\MPs-assets\improved_cleaned_data\state_year\bihar_2024.parquet
Saved 333 rows: d:\IITM\Academics\Sem 3\DCV\Projects\MPs-assets\improved_cleaned_data\state_year\gujarat_2014.parquet
Saved 371 rows: d:\IITM\Academics\Sem 3\DCV\Projects\MPs-assets\improved_cleaned_data\state_year\gujarat_2019.parquet
Saved 266 rows: d:\IITM\Academics\Sem 3\DCV\Projects\MPs-assets\i

,state,year,file,column,dtype_after_readback
0,Andhra,2014,andhra_2014.parquet,State,str
1,Andhra,2014,andhra_2014.parquet,Election Year,int64
2,Andhra,2014,andhra_2014.parquet,Sno.,int64
3,Andhra,2014,andhra_2014.parquet,Candidate,string
4,Andhra,2014,andhra_2014.parquet,Party,string
...,...,...,...,...,...
238,Gujarat,2024,gujarat_2024.parquet,Criminal_Cases_parse_status,str
239,Gujarat,2024,gujarat_2024.parquet,duplicate_record_flag,bool
240,Gujarat,2024,gujarat_2024.parquet,Total_Assets_INR_iqr_outlier_flag,bool
241,Gujarat,2024,gujarat_2024.parquet,Liabilities_INR_iqr_outlier_flag,bool


Saved 3783 rows across 9 state/year Parquet files.
Manual review queue: 892


## Review and analysis notes

This dataset compares each election's candidate pool; it does not track the same people across elections. The normalized name is a matching aid only. Duplicate-looking rows are flagged because the MyNeta files do not provide a stable candidate ID or constituency.

The 1.5×IQR rule flags possible outliers within each state and year; asset distributions are skewed, so a high value is not proof of an error. Verify flagged values against the original page/affidavit. Record any correction separately with the old value, new value, evidence, reason and reviewer.

The notebook reports missingness counts but cannot establish MCAR/MAR/MNAR by itself. Investigate whether missingness varies by state, year or observed fields before arguing a mechanism. Do not impute financial values without evidence. For winner/loser comparisons, add/verify constituency and join the winner table by year, state and constituency, using the name key as a reviewed check.